<a href="https://colab.research.google.com/github/0301sdh/faceguard/blob/main/notebooks/server_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 9 ① 서버용 보호 코드 확인
노트북에 흩어져 있던 방식 C'를 `server/protect.py` 하나로 옮겼다. 이 노트북은 두 가지를 한다.
1. 서버가 쓸 모델 파일 3개를 Drive `faceguard/models/`에 만든다 (Mac에서 내려받아 `models/`에 넣음)
2. 옮긴 코드로 내 사진 4장을 보호해서, Week 8 노트북 결과(`week8/full_cmin_eps5_seed0`)와 같은지 비교한다

GPU(T4) 런타임에서 위에서부터 순서대로 실행

In [ ]:
# [준비] 패키지 설치 + FaceGuard 코드 받기 (insightface는 설치에 1 ~ 2분)
!pip install -q insightface onnx onnx2torch onnxruntime
import os
if os.path.exists('/content/faceguard'):
    !git -C /content/faceguard pull -q
else:
    !git clone -q https://github.com/0301sdh/faceguard /content/faceguard

from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/faceguard'
MODELS = f'{DRIVE}/models'
os.makedirs(MODELS, exist_ok=True)
print('준비 완료')

In [ ]:
# [모델 파일 만들기] 이미 있으면 건너뜀
import glob, sys, shutil
import numpy as np, torch, torch.nn.functional as F, onnxruntime as ort

# 1) 얼굴 찾기 (SimSwap이 쓰는 것)
if not os.path.exists(f'{MODELS}/scrfd_10g_bnkps.onnx'):
    !wget -q https://github.com/deepinsight/insightface/releases/download/v0.7/antelopev2.zip -O /content/antelopev2.zip
    !unzip -o -q /content/antelopev2.zip -d /content/antelopev2
    shutil.copy(glob.glob('/content/antelopev2/**/scrfd_10g_bnkps.onnx', recursive=True)[0], MODELS)

# 2) FaceID ArcFace
if not os.path.exists(f'{MODELS}/w600k_r50.onnx'):
    !wget -q https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_l.zip -O /content/buffalo_l.zip
    !unzip -o -q /content/buffalo_l.zip -d /content/buffalo_l
    shutil.copy(glob.glob('/content/buffalo_l/**/w600k_r50.onnx', recursive=True)[0], MODELS)

# 3) SimSwap ArcFace: 모델 객체 통째로 저장된 파일(.tar) → .onnx (SimSwap 코드 없이도 쓸 수 있게)
SW_ONNX = f'{MODELS}/simswap_arcface.onnx'
if not os.path.exists(SW_ONNX):
    if not os.path.exists('/content/SimSwap'):
        !git clone -q https://github.com/neuralchen/SimSwap /content/SimSwap
    !wget -q -nc -P /content/SimSwap/arcface_model https://github.com/neuralchen/SimSwap/releases/download/1.0/arcface_checkpoint.tar
    sys.path.insert(0, '/content/SimSwap')            # 저장된 객체가 SimSwap의 models.arcface_models를 찾음
    arc = torch.load('/content/SimSwap/arcface_model/arcface_checkpoint.tar', map_location='cpu', weights_only=False).eval()
    dummy = torch.rand(1, 3, 112, 112)
    torch.onnx.export(arc, dummy, SW_ONNX, input_names=['input'], output_names=['embedding'],
                      dynamic_axes={'input': {0: 'batch'}, 'embedding': {0: 'batch'}}, opset_version=17, dynamo=False)
    # 변환 확인: PyTorch 원본 vs onnx
    x = torch.rand(2, 3, 112, 112)
    with torch.no_grad():
        a = arc(x)
    b = torch.from_numpy(ort.InferenceSession(SW_ONNX, providers=['CPUExecutionProvider']).run(None, {'input': x.numpy()})[0])
    print(f'SimSwap ArcFace 원본 vs onnx 유사도 {F.cosine_similarity(a, b).min().item():.5f}')

print(sorted(os.listdir(MODELS)))

In [ ]:
# [확인] server/protect.py로 4장 보호 → Week 8 노트북 결과와 비교
import random, cv2
sys.path.insert(0, '/content/faceguard/server')
import importlib, protect
importlib.reload(protect)
fg = protect.FaceGuard(MODELS)
print('device:', fg.device)

PHOTOS = f'{DRIVE}/photos'
OLD = f'{DRIVE}/week8/full_cmin_eps5_seed0'     # Week 8 노트북이 만든 C' 결과
OUT = f'{DRIVE}/week9/check'
os.makedirs(OUT, exist_ok=True)
names = ['person_a_1', 'person_a_2', 'person_a_3', 'person_a_4']
# Week 8 노트북 출력 (입력 단계 신원: SimSwap ArcFace, FaceID ArcFace)
NOTEBOOK = {'person_a_1': (-0.173, -0.006), 'person_a_2': (-0.031, -0.039),
            'person_a_3': (0.066, 0.204), 'person_a_4': (0.040, 0.227)}

def sw_id(img):
    crop, _ = fg.align(img)
    with torch.no_grad():
        return None if crop is None else fg.sw_embed(fg.to_tensor(crop))

def fid_id(img):
    # FaceID 방식: 사진 전체에서 얼굴 찾기 → 5점 정렬 112 → ArcFace
    M = fg.fid_matrix(img, pad=0)
    if M is None:
        return None
    a = cv2.warpAffine(img, M, (112, 112), borderValue=0.0)
    with torch.no_grad():
        return F.normalize(fg.fid_arc((fg.to_tensor(a) - 0.5) / 0.5), dim=1)

def psnr(a, b):
    mse = ((a.astype(np.float32) - b.astype(np.float32)) ** 2).mean()
    return 10 * np.log10(255 ** 2 / max(mse, 1e-10))

cos = lambda a, b: '-' if a is None or b is None else f'{F.cosine_similarity(a, b).item():.3f}'
random.seed(0); torch.manual_seed(0)     # 노트북과 같게: 처음 한 번만 정하고 4장을 이어서
for name in names:
    photo = cv2.imread(f'{PHOTOS}/{name}.jpg')
    prot, info = fg.protect(photo, seed=None)
    if prot is None:
        print(f'{name} | {info}'); continue
    cv2.imwrite(f'{OUT}/{name}.png', prot)
    small = cv2.resize(photo, info['size'], interpolation=cv2.INTER_AREA) if info['scale'] < 1 else photo
    old = cv2.imread(f'{OLD}/{name}.png')
    same = '-' if old is None or old.shape != prot.shape else f'{psnr(old, prot):.1f}dB (최대 차이 {np.abs(old.astype(int) - prot).max()})'
    sec = ' '.join(f'{k} {v:.1f}s' for k, v in info['seconds'].items())
    print(f'{name} {info["size"][0]}×{info["size"][1]} | 노트북 결과와 같은 정도 {same} '
          f'| 입력: SimSwap ArcFace {cos(sw_id(photo), sw_id(prot))} (노트북 {NOTEBOOK[name][0]:.3f}), '
          f'FaceID ArcFace {cos(fid_id(small), fid_id(prot))} (노트북 {NOTEBOOK[name][1]:.3f}) | {sec}')